In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript

sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
district_flow_name = {
    1: 'Bellevue (excluding downtown)',
    2: 'Bellevue Downtown',
    3: 'Kirkland',
    4: 'Redmond',
    5: 'Seattle (excluding Seattle downtown)',
    6: 'Seattle downtown',
    7: 'Rest',
}
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [ ]:
def pct_fmt(x):
    return 'nan' if pd.isna(x) else f'{x:,.2f}%'

def pct_compare_fmt(x):
    return 'nan' if pd.isna(x) else f'{x:,.1f}%'

## Trip Arrival Times by Hour

In [ ]:
def arr_time_by_hr(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey' 

    trip_ok_1 = data1['Trip'][['arrtm', 'trexpfac', 'travdist']].query('travdist > 0 and travdist < 200')
    trip_ok_3 = data3['Trip'][['arrtm', 'trexpfac', 'travdist']].query('travdist > 0 and travdist < 200')
    trip_ok_1 = trip_ok_1.reset_index()
    trip_ok_3 = trip_ok_3.reset_index()
    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()

    #Trip arrival time by hour
    trip_ok_1['hr'] = min_to_hour(trip_ok_1['arrtm'], 0)
    trip_ok_3['hr'] = min_to_hour(trip_ok_3['arrtm'], 0)
    trip_1_time = trip_ok_1[['hr', 'trexpfac']].groupby('hr').sum()['trexpfac']
    trip_3_time = trip_ok_3[['hr', 'trexpfac']].groupby('hr').sum()['trexpfac']
    trip_1_time_share = 100 * trip_1_time / trip_1_time.sum()
    trip_3_time_share = 100 * trip_3_time / trip_3_time.sum()
    trip_time = pd.DataFrame()
    trip_time[name1 + ' (%)'] = trip_1_time_share
    trip_time[name3 + ' (%)'] = trip_3_time_share
    trip_time = get_differences(trip_time, name1 + ' (%)', name3 + ' (%)', 2)
    trip_time = recode_index(trip_time, 'hr', 'Arrival Hour')

    trip_time = trip_time[[f'{name1} (%)', f'{name3} (%)', f'Difference ({name1} (%) - {name3} (%))']]
    
    display(trip_time.style.format({
        f'{name1} (%)': pct_fmt,
        f'{name3} (%)': pct_fmt,
        f'Difference ({name1} (%) - {name3} (%))': pct_compare_fmt,
    }))

    fig = px.bar(
        trip_time.reset_index(),
        x='Arrival Hour',
        y=[f'{name1} (%)', f'{name3} (%)'],
        barmode='group',
        labels={'index': 'Arrival Hour', 'value': 'Trip Arrival Time (hr)', 'variable': 'Source'},
        title=f'Trip Arrival Time (hr) ({tag})'
    )
    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Trip Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
arr_time_by_hr(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
arr_time_by_hr(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='PSRC Region')

## Tour Primary Destination Arrival Times by Hour

In [ ]:
def tour_pd_arr_time(data1=data_daysim, data2=data_survey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 

    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()
    tour_ok_2 = data2['Tour_cloned'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_2 = tour_ok_2.reset_index()

    #Tour Primary Destination arrival time by hour
    tour_ok_1['hrapd'] = min_to_hour(tour_ok_1['tardest'], 0)
    tour_ok_2['hrapd'] = min_to_hour(tour_ok_2['tardest'], 0)
    tour_1_time_apd = tour_ok_1[['hrapd', 'toexpfac']].groupby('hrapd').sum()['toexpfac']
    tour_2_time_apd = tour_ok_2[['hrapd', 'toexpfac']].groupby('hrapd').sum()['toexpfac']
    tour_1_time_share_apd = 100 * tour_1_time_apd / tour_1_time_apd.sum()
    tour_2_time_share_apd = 100 * tour_2_time_apd / tour_2_time_apd.sum()
    tour_time_apd = pd.DataFrame()
    tour_time_apd[name1 + ' (%)'] = tour_1_time_share_apd
    tour_time_apd[name2 + ' (%)'] = tour_2_time_share_apd
    tour_time_apd = get_differences(tour_time_apd, name1 + ' (%)', name2 + ' (%)', 2)
    tour_time_apd = recode_index(tour_time_apd, 'hrapd', 'Primary Destination Arrival Hour')
    tour_time_apd = tour_time_apd.loc[:, [f'{name1} (%)', f'{name2} (%)', f'Difference ({name1} (%) - {name2} (%))']]

    
    display(tour_time_apd.style.format({
        f'{name1} (%)': pct_fmt,
        f'{name2} (%)': pct_fmt,
        f'Difference ({name1} (%) - {name2} (%))': pct_compare_fmt,
    }))

    fig = px.bar(
        tour_time_apd.reset_index(),
        x='Primary Destination Arrival Hour',
        y=[f'{name1} (%)', f'{name2} (%)'],
        barmode='group',
        labels={'index': 'Primary Destination Arrival Hour', 'value': 'Tour Arrival Time (hr)', 'variable': 'Source'},
        title=f'Tour Arrival Time (hr) ({tag})'
    )

    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Tour Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tour_pd_arr_time(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tour_pd_arr_time(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Primary Destination Departure Times by Hour

In [ ]:
def tour_pd_depart_time(data1=data_daysim, data2=data_survey, tag='PSRC Region'):    
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 

    tour_ok_1 = data1['Tour'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_1 = tour_ok_1.reset_index()
    tour_ok_2 = data2['Tour_cloned'][['tardest', 'tlvdest', 'toexpfac', 'tautodist']].query('tautodist > 0 and tautodist < 200')
    tour_ok_2 = tour_ok_2.reset_index()

    #Tour Primary Destination Departure time by hour
    tour_ok_1['hrlpd'] = min_to_hour(tour_ok_1['tlvdest'], 0)
    tour_ok_2['hrlpd'] = min_to_hour(tour_ok_2['tlvdest'], 0)
    tour_1_time_lpd = tour_ok_1[['hrlpd', 'toexpfac']].groupby('hrlpd').sum()['toexpfac']
    tour_2_time_lpd = tour_ok_2[['hrlpd', 'toexpfac']].groupby('hrlpd').sum()['toexpfac']
    tour_1_time_share_lpd = 100 * tour_1_time_lpd / tour_1_time_lpd.sum()
    tour_2_time_share_lpd = 100 * tour_2_time_lpd / tour_2_time_lpd.sum()
    tour_time_lpd = pd.DataFrame()
    tour_time_lpd[name1 + ' (%)'] = tour_1_time_share_lpd
    tour_time_lpd[name2 + ' (%)'] = tour_2_time_share_lpd
    tour_time_lpd = get_differences(tour_time_lpd, name1 + ' (%)', name2 + ' (%)', 2)
    tour_time_lpd = recode_index(tour_time_lpd, 'hrlpd', 'Primary Destination Departure Hour')
    tour_time_lpd = tour_time_lpd.loc[:, [f'{name1} (%)', f'{name2} (%)', f'Difference ({name1} (%) - {name2} (%))']]

    display(tour_time_lpd.style.format({
        f'{name1} (%)': pct_fmt,
        f'{name2} (%)': pct_fmt,
        f'Difference ({name1} (%) - {name2} (%))': pct_compare_fmt,
    }))

    fig = px.bar(
        tour_time_lpd.reset_index(),
        x='Primary Destination Departure Hour',
        y=[f'{name1} (%)', f'{name2} (%)'],
        barmode='group',
        labels={'index': 'Primary Destination Departure Hour', 'value': 'Tour Departure Time (hr)', 'variable': 'Source'},
        title=f'Tour Departure Time (hr) ({tag})'
    )

    fig.update_yaxes(ticksuffix='%')
    fig.update_layout(xaxis_title='Hour of Day', 
                      yaxis_title='Tour Share (%)',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tour_pd_depart_time(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tour_pd_depart_time(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Model Bike Trips by Departure Time by Purpose (Daysim)

In [ ]:
def bike_trip_depart_time_by_purp(data1=data_daysim, tag='PSRC Region'):    
    ### Create bike trip distribution by departure time and purpose
    bike_trips_df = data1['Trip'][['deptm', 'trexpfac', 'mode', 'dpurp']].query('mode == "Bike"').reset_index()
    bike_trips_df['hr'] = min_to_hour(bike_trips_df['deptm'], 0)
    bike_trips_by_hr_df = bike_trips_df[['hr', 'trexpfac']].groupby('hr').sum().reset_index()
    bike_trips_by_hr_df.rename(columns = {'hr':'Departure Hour', 'trexpfac':'All Purposes'}, inplace = True)
    trip_purposes = bike_trips_df['dpurp'].unique()
    for purp in trip_purposes:
        bike_trip_purp_by_hf_df = bike_trips_df.loc[bike_trips_df['dpurp'] == purp, ['hr', 'trexpfac']].groupby('hr').sum().reset_index()
        bike_trip_purp_by_hf_df.rename(columns = {'trexpfac':purp}, inplace = True)
        bike_trips_by_hr_df = bike_trips_by_hr_df.merge(bike_trip_purp_by_hf_df, left_on = 'Departure Hour', right_on = 'hr', how = 'outer')
        bike_trips_by_hr_df = bike_trips_by_hr_df.drop(columns = ['hr'])
    
    # table
    display(bike_trips_by_hr_df.style.format({col: '{:,.0f}' for col in bike_trips_by_hr_df.columns if col != 'Departure Hour'}))

In [ ]:
bike_trip_depart_time_by_purp(data1=data_daysim, tag='PSRC Region')

In [ ]:
bike_trip_depart_time_by_purp(data1=data_daysim_bkr, tag='BKR')

## Model Bike Tours Departure Time by Purpose (Daysim)

In [ ]:
def bike_tour_depart_time_by_purp(data1=data_daysim, tag='PSRC Region'):
    ### create bike tour distribution by departure time and purpose
    bike_tours_df = data1['Tour'][['tlvorig', 'toexpfac', 'tmodetp', 'pdpurp']].query('tmodetp == "Bike"').reset_index()
    bike_tours_df['hrlvo'] = min_to_hour(bike_tours_df['tlvorig'], 0)
    bike_tours_by_hr_df = bike_tours_df[['hrlvo', 'toexpfac']].groupby('hrlvo').sum().reset_index()
    bike_tours_by_hr_df.rename(columns = {'hrlvo':'Departure Hour', 'toexpfac':'All Purposes'}, inplace = True)
    tour_purposes = bike_tours_df['pdpurp'].unique()
    for purp in tour_purposes:
        bike_tour_purp_by_hr_df = bike_tours_df.loc[bike_tours_df['pdpurp'] == purp, ['hrlvo', 'toexpfac']].groupby('hrlvo').sum().reset_index()
        bike_tour_purp_by_hr_df.rename(columns = {'toexpfac': purp}, inplace = True)
        bike_tours_by_hr_df = bike_tours_by_hr_df.merge(bike_tour_purp_by_hr_df, left_on = 'Departure Hour', right_on = 'hrlvo', how = 'outer')
        bike_tours_by_hr_df = bike_tours_by_hr_df.drop(columns = ['hrlvo'])
    
    # table
    display(bike_tours_by_hr_df.style.format({col: '{:,.0f}' for col in bike_tours_by_hr_df.columns if col != 'Departure Hour'}))

In [ ]:
bike_tour_depart_time_by_purp(data1=data_daysim, tag='PSRC Region')

In [ ]:
bike_tour_depart_time_by_purp(data1=data_daysim_bkr, tag='BKR')